# 03 · Flipkart Scraper  *(Task 3)*

**Goal:** collect live advertised prices from Flipkart for chosen search terms (laptop, printer, ink …): price, seller name, product id,
product link and a screenshot of the product page.

**How it works:** Colab starts a hidden ("headless") Chrome browser controlled by Selenium, opens the Flipkart search page, reads each product card,
then opens each product page to find the seller and take a screenshot. Results are saved as a CSV in `scraper_output/`.

**Things to know before running**
* Amazon is intentionally **not** scraped (it blocks automated access and its terms prohibit it). Only Flipkart is covered.
* Websites change their page layout. Flipkart's internal CSS class names (e.g. `k7wcnx`) can change at any time; if the scraper finds 0 products, use the diagnostic cell at the bottom, then update the class names in the settings cell.
* Flipkart's product id (the `pid=` in the link) is **not** the manufacturer SKU. The last cells measure how many scraped ids match your SKU master and only
  hand matching rows to the monitoring pipeline. Expect a low match rate until a Flipkart-id → SKU mapping exists.
* Be polite: keep the pauses, and check Flipkart's terms of use for your use case.

**Needs:** notebook 00 run. Notebook 02 run (only for the final matching step). This notebook needs internet access and cannot run without a real browser, so it works only in Colab.

**What this cell does:** Connects Google Drive, finds your project folder and loads the shared helper module `map_common`.

**Purpose:** Every notebook starts here so they all use the same folders, database settings and helpers.

In [ ]:
# ---- Where the project lives on Google Drive (change only if you used a different folder name) ----
PROJECT_DIR = "/content/drive/MyDrive/MAP_Compliance_Project"

import os, sys, importlib
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running inside Colab - using the folder path as given.")

if not os.path.exists(os.path.join(PROJECT_DIR, "map_common.py")):
    raise FileNotFoundError("map_common.py not found. Run notebook 00_Project_Setup first "
                            "(and check PROJECT_DIR above matches your Drive folder).")
sys.path.insert(0, PROJECT_DIR)
import map_common as mc
importlib.reload(mc)
mc.ensure_dirs()
print("Project folder:", mc.PROJECT_DIR)

**What this cell does:** Defines the scraper settings: which search terms to run and how many products to take for each, how long to wait for pages to load, and the page-layout class names the scraper looks for.

**Purpose:** Everything you might want to change (what to search, how many items, waits, class names) is in one place, away from the scraping logic.

In [ ]:
QUERIES = {"laptop": 5, "printer": 5, "ink": 5}   # search term -> how many products to collect
MARKETPLACE, REGION = "Flipkart", "India"
SEARCH_WAIT_SEC, DETAIL_WAIT_SEC = 3, 2           # pauses so pages finish loading (and we stay polite)
LIST_CARD_CLASS = "k7wcnx"                        # product card on list-style results (e.g. laptops)
GRID_CARD_CLASS = "fb4uj3"                        # product card on grid-style results (e.g. printers, ink)
PRICE_CLASS     = "hZ3P6w"                        # the price box inside a card
COPY_TO_SELLER_DATA = False                       # True = also write SKU-matched rows to seller_data/<today>.csv
print("Searches:", QUERIES)

**What this cell does:** Defines `parse_price()`, which turns text such as `₹45,990` or `Rs. 1,299.50` into the number 45990.0 or 1299.5, and runs a few self-tests.

**Purpose:** Prices come from web pages as text. The monitoring logic needs real numbers, so the conversion is done here, once, and tested without needing a browser.

In [ ]:
import re

def parse_price(text):
    """'₹45,990' -> 45990.0 ; returns None if no number is found. Takes the FIRST price if two are shown."""
    if text is None:
        return None
    m = re.search(r"(\d[\d,]*(?:\.\d+)?)", str(text))
    return float(m.group(1).replace(",", "")) if m else None

tests = {"₹45,990": 45990.0, "Rs. 1,299.50": 1299.5, "₹999": 999.0, "₹1,23,456": 123456.0,
         "₹45,990 ₹52,000": 45990.0, "Out of stock": None, None: None, "": None}
for given, expected in tests.items():
    got = parse_price(given)
    assert got == expected, f"parse_price({given!r}) gave {got}, expected {expected}"
print("parse_price self-tests passed:", len(tests))

**What this cell does:** Installs Google Chrome and the Selenium libraries inside Colab (skipped if already present).

**Purpose:** The scraper needs a real browser engine. Colab does not include one, so it is installed on demand (about 1 minute).

In [ ]:
import shutil, subprocess, sys
if shutil.which("google-chrome-stable") is None:
    print("Installing Google Chrome (first time in this session)...")
    subprocess.run("wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb -O /tmp/chrome.deb "
                   "&& apt-get install -y -qq /tmp/chrome.deb", shell=True, check=True)
mc.ensure_packages({"selenium": "selenium", "webdriver_manager": "webdriver-manager", "bs4": "beautifulsoup4"})
print(subprocess.run(["google-chrome-stable", "--version"], capture_output=True, text=True).stdout.strip())

**What this cell does:** Starts the hidden Chrome browser (no window, sandbox disabled as Colab requires) and checks it can open Flipkart.

**Purpose:** Creates the single browser session reused by every search. The title test shows early whether Flipkart is reachable or blocking us.

In [ ]:
import os, time
from datetime import datetime
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

opts = Options()
for arg in ("--headless=new", "--no-sandbox", "--disable-dev-shm-usage", "--window-size=1920,1080"):
    opts.add_argument(arg)
opts.binary_location = shutil.which("google-chrome-stable")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)

SHOT_DIR = os.path.join(mc.SCRAPER_DIR, "screenshots")
os.makedirs(SHOT_DIR, exist_ok=True)
driver.get("https://www.flipkart.com/search?q=laptop")
print("Browser ready. Page title:", driver.title)

**What this cell does:** Defines `scrape_flipkart(query, max_products)`: opens the search page, finds product cards (list layout first, then grid layout, then a generic fallback), and for each product opens its page to read the seller name and save a screenshot.

**Purpose:** This is the core of Task 3. Each product becomes one record with price, seller, id, link, screenshot and the time it was collected. A failure on one product is reported and skipped, never stops the run.

In [ ]:
def find_cards(soup):
    cards = soup.find_all("a", class_=LIST_CARD_CLASS)
    if cards:
        return cards, "list"
    cards = soup.find_all("a", class_=GRID_CARD_CLASS)
    if cards:
        return cards, "grid"
    # Fallback when Flipkart renames its classes: any product link that shows a rupee price
    cards = [a for a in soup.find_all("a", href=True) if "pid=" in a["href"] and "\u20b9" in a.get_text()]
    return cards, "fallback"

def scrape_flipkart(query, max_products=5):
    driver.get("https://www.flipkart.com/search?q=" + query.replace(" ", "%20"))
    time.sleep(SEARCH_WAIT_SEC)
    cards, layout = find_cards(BeautifulSoup(driver.page_source, "html.parser"))
    cards = cards[:max_products]
    print(f"[{query}] layout: {layout} | products found: {len(cards)}")

    records = []
    for i, card in enumerate(cards):
        try:
            href = card.get("href")
            url = href if href.startswith("http") else "https://www.flipkart.com" + href
            price_div = card.find("div", class_=PRICE_CLASS)
            price_text = price_div.text.strip() if price_div else None
            if price_text is None:                                 # fallback: first rupee amount in the card text
                m = re.search(r"\u20b9\s*[\d,]+(?:\.\d+)?", card.get_text(" "))
                price_text = m.group(0) if m else None
            pid = re.search(r"pid=([A-Z0-9]+)", href)

            driver.get(url)                                        # seller is only shown on the product page
            time.sleep(DETAIL_WAIT_SEC)
            detail = BeautifulSoup(driver.page_source, "html.parser")
            seller_node = detail.find(string=re.compile(r"^Seller:\s*\S"))
            seller = seller_node.strip().replace("Seller:", "").strip() if seller_node else None

            stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            shot = os.path.join(SHOT_DIR, f"flipkart_{re.sub(r'[^a-z0-9]+', '_', query.lower())}_{i}_{stamp}.png")
            driver.save_screenshot(shot)

            records.append({"Seller_name": seller, "SKU": pid.group(1) if pid else None, "Region": REGION,
                            "Advertised_price_text": price_text, "Advertised_price": parse_price(price_text),
                            "Marketplace": MARKETPLACE, "Screenshot": os.path.relpath(shot, mc.PROJECT_DIR),
                            "Live_link": url, "Last_time_collected": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})
        except Exception as exc:
            print(f"  skipped product {i}: {exc}")
    return pd.DataFrame(records)
print("Scraper function defined.")

**What this cell does:** Runs the scraper for every search term in the settings, combines the results, saves them to a time-stamped CSV in `scraper_output/`, and always closes the browser, even if something fails.

**Purpose:** Produces the Task 3 deliverable: the raw scraped price data. Closing the browser in a `finally` block avoids leaving Chrome running and using Colab memory.

In [ ]:
frames = []
try:
    for q, n in QUERIES.items():
        frames.append(scrape_flipkart(q, n))
finally:
    driver.quit()
    print("Browser closed.")

df_scrape = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if df_scrape.empty:
    print("No products were collected. Run the diagnostic cell at the bottom of this notebook.")
else:
    out_csv = os.path.join(mc.SCRAPER_DIR, f"flipkart_scrape_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
    df_scrape.to_csv(out_csv, index=False)
    print(f"Saved {len(df_scrape)} rows -> {os.path.relpath(out_csv, mc.PROJECT_DIR)}")
    print("Missing seller:", int(df_scrape['Seller_name'].isna().sum()), "| missing price:", int(df_scrape['Advertised_price'].isna().sum()))
df_scrape

**What this cell does:** Reshapes the scraped rows into the same column layout as the daily seller files (SKU, Marketplace, Seller_name, region, adv_price, Screenshot, live_link) and measures how many scraped product ids exist in your SKU master (`cleaned_data/SKU_TABLE.csv`).

**Purpose:** Shows honestly whether the scraped data can feed the monitoring pipeline. Only rows whose id matches a known SKU can be judged against MAP/LPP; the rest are saved separately so nothing is lost.

In [ ]:
if df_scrape.empty:
    print("Nothing to convert.")
else:
    seller_fmt = pd.DataFrame({
        "SKU": df_scrape["SKU"], "Marketplace": df_scrape["Marketplace"], "Seller_name": df_scrape["Seller_name"],
        "region": df_scrape["Region"], "adv_price": df_scrape["Advertised_price"],
        "Screenshot": df_scrape["Screenshot"], "live_link": df_scrape["Live_link"]})
    sku_csv = os.path.join(mc.CLEAN_DIR, "SKU_TABLE.csv")
    if os.path.exists(sku_csv):
        known = set(pd.read_csv(sku_csv)["SKU"].astype(str))
        seller_fmt["matches_sku_master"] = seller_fmt["SKU"].astype(str).isin(known)
        rate = seller_fmt["matches_sku_master"].mean()
        print(f"{int(seller_fmt['matches_sku_master'].sum())} of {len(seller_fmt)} scraped rows ({rate:.0%}) match a SKU in the master list.")
        if rate < 0.5:
            print("Low match: Flipkart product ids are not manufacturer SKUs. A Flipkart-id -> SKU mapping is needed for full automation.")
    else:
        seller_fmt["matches_sku_master"] = False
        print("cleaned_data/SKU_TABLE.csv not found - run notebook 02 first to measure the match rate.")

    today = datetime.now().strftime("%Y-%m-%d")
    all_path = os.path.join(mc.SCRAPER_DIR, f"flipkart_seller_format_{today}.csv")
    seller_fmt.to_csv(all_path, index=False)
    print("Saved (all rows, seller-file layout):", os.path.relpath(all_path, mc.PROJECT_DIR))

    if COPY_TO_SELLER_DATA:
        target = os.path.join(mc.SELLER_DIR, f"{today}.csv")
        matched = seller_fmt[seller_fmt["matches_sku_master"]].drop(columns="matches_sku_master")
        if os.path.exists(target):
            print(f"Not copied: {os.path.relpath(target, mc.PROJECT_DIR)} already exists (never overwritten).")
        elif matched.empty:
            print("Not copied: no scraped row matches a known SKU.")
        else:
            matched.to_csv(target, index=False)
            print(f"Copied {len(matched)} matching rows to", os.path.relpath(target, mc.PROJECT_DIR))
seller_fmt.head() if not df_scrape.empty else None

### Diagnostic (only if the scraper finds 0 products)
Run the next two cells **in a fresh session** (the browser was closed above). They load a search page, show a screenshot (to spot a CAPTCHA or block) and print the class names of the elements around each price so you can update `LIST_CARD_CLASS`, `GRID_CARD_CLASS` and `PRICE_CLASS`.

**What this cell does:** Restarts a browser, opens a Flipkart search, prints the page title and size, and displays a screenshot.

**Purpose:** If the screenshot shows a CAPTCHA or an error page, the problem is blocking, not class names.

In [ ]:
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)
driver.get("https://www.flipkart.com/search?q=laptop"); time.sleep(3)
print("Title:", driver.title, "| page length:", len(driver.page_source))
dbg = os.path.join(mc.SCRAPER_DIR, "debug_page.png"); driver.save_screenshot(dbg)
from IPython.display import Image, display
display(Image(dbg))

**What this cell does:** Finds elements that look like a single price (e.g. ₹45,990) and prints the class names of the price box and of its parent elements up to the enclosing link; then closes the browser.

**Purpose:** Tells you which class names Flipkart uses today, so you can paste them into the settings cell.

In [ ]:
soup = BeautifulSoup(driver.page_source, "html.parser")
prices = [d for d in soup.find_all("div") if d.get("class") and re.fullmatch(r"\u20b9[\d,]+", d.text.strip())]
print("Price boxes found:", len(prices))
if prices:
    node = prices[0]; print("PRICE_CLASS candidate:", node.get("class"))
    for level in range(8):
        node = node.parent
        if node is None: break
        print(f"level {level}: <{node.name}> class={node.get('class')} href={node.get('href') if node.name == 'a' else None}")
driver.quit()